# Doc-Parser Quickstart
PP-DocLayout-V3 + GLM-OCR Document Parsing Pipeline

This notebook demonstrates the complete two-stage document parsing pipeline:
1. **PP-DocLayout-V3** (server-side): Layout detection with 23 categories
2. **GLM-OCR 0.9B** (server-side): Text/table/formula recognition

All heavy computation runs on the Z.AI MaaS API — no GPU required.

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import json
from pathlib import Path

# Set your Z.AI API key (or use .env file)
# os.environ["Z_AI_API_KEY"] = "your-key-here"

from doc_parser.config import get_settings, configure_logging
configure_logging("INFO")

try:
    settings = get_settings()
    print(f"\u2713 Settings loaded")
    print(f"  config: {settings.config_yaml_path}")
    print(f"  output: {settings.output_dir}")
except Exception as e:
    print(f"\u26a0 Settings error: {e}")
    print("  Set Z_AI_API_KEY in .env or os.environ to run API calls")

In [ ]:
from doc_parser.pipeline import DocumentParser, ParseResult

# Replace with your document path
sample_path = Path("../data/raw/sample.pdf")  # or sample.png

if not sample_path.exists():
    print(f"\u26a0 Sample file not found at {sample_path}")
    print("  Place a PDF or image in data/raw/ and update the path above")
else:
    parser = DocumentParser()
    result = parser.parse_file(sample_path)

    print(f"\u2713 Parsed: {sample_path.name}")
    print(f"  Pages: {len(result.pages)}")
    print(f"  Total elements: {result.total_elements}")
    for page in result.pages:
        print(f"  Page {page.page_num}: {len(page.elements)} elements")

In [ ]:
from IPython.display import Markdown, display

if 'result' in dir():
    full_markdown = "\n\n".join(p.markdown for p in result.pages if p.markdown)
    display(Markdown(full_markdown))
else:
    # Demo: show what markdown output looks like
    demo_md = """
# Sample Document Title

**Abstract:** This is a sample document demonstrating the PP-DocLayout-V3 + GLM-OCR pipeline.

## 1. Introduction

The pipeline detects 23 layout categories and recognizes text, tables, and formulas.

## 2. Mathematical Content

$$
E = mc^2
$$

| Method | F1 Score | Notes |
|--------|----------|-------|
| PP-DocLayout-V3 + GLM-OCR | 94.62 | OmniDocBench V1.5 #1 |
"""
    display(Markdown(demo_md))
    print("(Demo output \u2014 parse a real document to see actual results)")

In [ ]:
from doc_parser.chunker import structure_aware_chunking

if 'result' in dir():
    all_chunks = []
    for page in result.pages:
        chunks = structure_aware_chunking(
            page.elements,
            source_file=Path(result.source_file).name,
            page=page.page_num,
        )
        all_chunks.extend(chunks)

    print(f"\u2713 Generated {len(all_chunks)} RAG-ready chunks")
    print()
    for i, chunk in enumerate(all_chunks[:3]):  # show first 3
        print(f"Chunk {i+1}:")
        print(f"  ID: {chunk.chunk_id}")
        print(f"  Types: {chunk.element_types}")
        print(f"  Atomic: {chunk.is_atomic}")
        print(f"  Text preview: {chunk.text[:100]}...")
        print()
else:
    # Demo chunk output
    print("Demo chunk output:")
    print("  Chunk 1: {'chunk_id': 'sample.pdf_1_0', 'element_types': ['document_title', 'paragraph'], 'is_atomic': False}")
    print("  Chunk 2: {'chunk_id': 'sample.pdf_1_1', 'element_types': ['table'], 'is_atomic': True}")
    print("  (Parse a real document to see actual chunks)")

In [ ]:
from PIL import Image, ImageDraw

if 'result' in dir() and Path(result.source_file).suffix.lower() in {'.png', '.jpg', '.jpeg'}:
    # For image inputs, visualize bboxes directly
    img = Image.open(result.source_file).convert("RGB")
    draw = ImageDraw.Draw(img)

    COLORS = {
        "document_title": "red",
        "paragraph_title": "blue",
        "paragraph": "green",
        "table": "orange",
        "formula": "purple",
        "image": "gray",
    }

    for page in result.pages:
        for el in page.elements:
            if el.bbox and len(el.bbox) == 4:
                w, h = img.size
                x1, y1, x2, y2 = [
                    int(el.bbox[0] * w), int(el.bbox[1] * h),
                    int(el.bbox[2] * w), int(el.bbox[3] * h)
                ]
                color = COLORS.get(el.label, "yellow")
                draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

    from IPython.display import display as ipy_display
    ipy_display(img.resize((800, int(800 * img.height / img.width))))
elif 'result' in dir():
    print("Bbox visualization is available for image inputs (.png, .jpg)")
    print("For PDFs, use doc_parser.utils.pdf_utils.pdf_page_to_image() first")
else:
    print("Parse a document first to visualize layout regions")
    print("Example: pdf_page_to_image(Path('doc.pdf'), page_num=0, dpi=72)")